## Notebook to Create Dataset of Prompts. 

In [ ]:
def default_params(): 
    return {
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/pipeline/curated',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'prompts' : {
            'P1' : lambda code: f"{code}",
            'P2' : lambda code: f"Complete the following code:\n{code}",
            'P3' : lambda code: f"You are an expert software engineer who writes clean, maintainable, and production-quality code. Your goal is to complete the following code, with high readability and without introducing any common code smells:\n{code}",
            'P4' : lambda code: 
            f"""You are an expert software engineer who writes clean, maintainable, and production-quality code. Your goal is to complete the provided code with high readability and without introducing any common code smells such as: 
            - Long functions
            - Deeply nested logic
            - Duplicated code
            - Hardcoded constants
            - Poor naming conventions
            - Unused variables
            Please follow these principles:
            1. Write small, focused functions.
            2. Use clear and descriptive variable and function names.
            3. Avoid deeply nested conditionals or loops.
            4. Use constants or configuration instead of magic numbers.
            5. Follow best practices in PYTHON.

            Complete the following code:\n{code}""",
        },
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'output_dir': '/workspaces/CodeSmells/semeru-datasets/code_smells/prompts'
    }
params = default_params()


### Imports

In [2]:
import pandas as pd
import os

In [3]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Dataset loading

In [4]:
df_dataset = pd.read_json(f"{params['dataset']['path']}_{params['dataset']['sampling_size']}.json")

### Create prompt Datasets

In [ ]:
def create_prompt_dataset(prompt_id, df_dataset):
    df_prompt = df_dataset.copy()
    df_prompt['prompt_id'] = prompt_id
    df_prompt[params['dataset']['prompt_column']] = params['prompts'][prompt_id]('')
    df_prompt['original_code'] = df_prompt[params['dataset']['content_column']]
    df_prompt[params['dataset']['content_column']] = df_prompt['original_code'].apply(params['prompts'][prompt_id])
    return df_prompt

In [6]:
def create_prompts(df_dataset):
    for prompt_id in params['prompts'].keys():
        df_prompt = create_prompt_dataset(prompt_id, df_dataset)
        create_folder(params['output_dir'])
        output_path = f"{params['output_dir']}/{prompt_id}_{params['dataset']['sampling_size']}.json"
        df_prompt.to_json(output_path)
        print(f"Prompt dataset for {prompt_id} saved to {output_path}")

In [7]:
create_prompts(df_dataset)

Prompt dataset for P1 saved to /workspaces/CodeSmells/semeru-datasets/code_smells/prompts/P1_500.json
Prompt dataset for P2 saved to /workspaces/CodeSmells/semeru-datasets/code_smells/prompts/P2_500.json
Prompt dataset for P3 saved to /workspaces/CodeSmells/semeru-datasets/code_smells/prompts/P3_500.json
Prompt dataset for P4 saved to /workspaces/CodeSmells/semeru-datasets/code_smells/prompts/P4_500.json
